# ⚠️ Comprehensive Risk Assessment
## Quantitative Trading Strategy MVP Project

This notebook performs comprehensive risk analysis and generates institutional-grade risk reports.


In [ ]:
# Setup and imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
from datetime import datetime, timedelta
from scipy import stats
import pickle
import sys
import asyncio
warnings.filterwarnings('ignore')

# Windows: ensure selector event loop policy for ZMQ compatibility
if sys.platform.startswith('win'):
    try:
        from asyncio import WindowsSelectorEventLoopPolicy
        asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())
    except Exception:
        pass

# Import custom modules
sys.path.append('../src')
from models.risk_models import RiskModels
from risk_management.portfolio_risk import PortfolioRiskManager
from backtesting.performance_metrics import PerformanceAnalyzer

print("⚠️ Risk assessment setup complete!")


In [ ]:
# Load backtest results
try:
    with open('../data/comprehensive_backtest_results.pkl', 'rb') as f:
        backtest_data = pickle.load(f)
    with open('../data/executive_summary.pkl', 'rb') as f:
        executive_summary = pickle.load(f)

    backtest_results = backtest_data['backtest_results']
    print("✅ Loaded comprehensive backtest results")
    print(f"   Available strategies: {list(backtest_results.keys())}")

    best_strategy_name = executive_summary.get('best_strategy', {}).get('name')
    if not best_strategy_name:
        best_strategy_name = list(backtest_results.keys())[0]

except FileNotFoundError:
    print("⚠️ Backtest results not found. Creating sample data for risk analysis...")
    np.random.seed(42)
    dates = pd.date_range('2020-01-01', '2023-12-31', freq='D')
    strategy_returns = pd.Series(np.random.normal(0.0008, 0.015, len(dates)), index=dates)
    backtest_results = {
        'sample_strategy': {
            'detailed_results': {
                'portfolio_history': pd.DataFrame({
                    'Portfolio_Value': (1 + strategy_returns).cumprod() * 100000,
                    'Returns': strategy_returns
                }, index=dates)
            }
        }
    }
    best_strategy_name = 'sample_strategy'

# Initialize risk tools
risk_models = RiskModels()
portfolio_risk_manager = PortfolioRiskManager()
performance_analyzer = PerformanceAnalyzer()

print("\n📊 Ready for risk analysis")


## 2. Advanced Value at Risk (VaR) Analysis


In [ ]:
# Extract strategy returns
best_strategy_data = backtest_results[best_strategy_name]
portfolio_history = best_strategy_data['detailed_results']['portfolio_history']
strategy_returns = portfolio_history['Returns'].dropna()

print(f"📊 Advanced VaR Analysis for {best_strategy_name}:")
print(f"   Data period: {strategy_returns.index[0].date()} to {strategy_returns.index[-1].date()}")
print(f"   Total observations: {len(strategy_returns)}")

var_results = {}
confidence_levels = [0.95, 0.99, 0.995]
for confidence_level in confidence_levels:
    print(f"\n📈 VaR Analysis at {confidence_level:.1%} confidence level:")
    hist_var = portfolio_risk_manager.historical_var(strategy_returns, confidence_level)
    param_var = portfolio_risk_manager.parametric_var(strategy_returns, confidence_level)
    mc_var = portfolio_risk_manager.monte_carlo_var(strategy_returns, confidence_level)
    cvar = portfolio_risk_manager.conditional_var(strategy_returns, confidence_level)
    evt = risk_models.extreme_value_theory_var(strategy_returns, confidence_level)
    evt_var = evt.get('var_evt', np.nan)
    evt_es = evt.get('es_evt', np.nan)
    var_results[confidence_level] = {
        'historical_var': hist_var,
        'parametric_var': param_var,
        'monte_carlo_var': mc_var,
        'conditional_var': cvar,
        'evt_var': evt_var,
        'evt_es': evt_es
    }
    print(f"   Historical VaR: {hist_var:.4f} ({hist_var*100:.2f}%)")
    print(f"   Parametric VaR: {param_var:.4f} ({param_var*100:.2f}%)")
    print(f"   Monte Carlo VaR: {mc_var:.4f} ({mc_var*100:.2f}%)")
    print(f"   Conditional VaR: {cvar:.4f} ({cvar*100:.2f}%)")
    if not np.isnan(evt_var):
        print(f"   EVT VaR: {evt_var:.4f} ({evt_var*100:.2f}%)")
        print(f"   EVT Expected Shortfall: {evt_es:.4f} ({evt_es*100:.2f}%)")

var_comparison = []
for conf_level, results in var_results.items():
    for method, value in results.items():
        if not np.isnan(value):
            var_comparison.append({
                'Confidence_Level': f"{conf_level:.1%}",
                'Method': method.replace('_', ' ').title(),
                'VaR_Value': value,
                'VaR_Percent': value * 100,
                'Dollar_Amount': value * portfolio_history['Portfolio_Value'].iloc[-1]
            })
var_df = pd.DataFrame(var_comparison)
print("\n📋 VaR Comparison Summary:")
display(var_df.round(4))
fig = px.bar(var_df, x='Method', y='VaR_Percent', color='Confidence_Level', title='Value at Risk Comparison Across Methods', labels={'VaR_Percent': 'VaR (%)', 'Method': 'VaR Method'}, barmode='group')
fig.show()

def var_backtest(returns, var_estimate, confidence_level):
    violations = (returns < -var_estimate).sum()
    total_obs = len(returns)
    expected_violations = total_obs * (1 - confidence_level)
    violation_rate = violations / total_obs
    if violations > 0 and violations < total_obs:
        lr_stat = 2 * (violations * np.log(violation_rate / (1 - confidence_level)) + (total_obs - violations) * np.log((1 - violation_rate) / confidence_level))
        p_value = 1 - stats.chi2.cdf(lr_stat, 1)
    else:
        lr_stat = np.inf
        p_value = 0
    return {
        'violations': violations,
        'expected_violations': expected_violations,
        'violation_rate': violation_rate,
        'lr_statistic': lr_stat,
        'p_value': p_value,
        'test_passed': p_value > 0.05
    }

print("\n🧪 VaR Model Validation (Backtesting):")
hist_var_95 = var_results[0.95]['historical_var']
backtest_results_var = var_backtest(strategy_returns, hist_var_95, 0.95)
print("   Historical VaR (95%) Backtesting:")
print(f"     Actual violations: {backtest_results_var['violations']}")
print(f"     Expected violations: {backtest_results_var['expected_violations']:.1f}")
print(f"     Violation rate: {backtest_results_var['violation_rate']:.2%}")
print(f"     Kupiec test p-value: {backtest_results_var['p_value']:.4f}")
print(f"     Model validation: {'✅ PASSED' if backtest_results_var['test_passed'] else '❌ FAILED'}")


## 3. Stress Testing and Scenario Analysis


In [4]:
# Define comprehensive stress testing scenarios
stress_scenarios = {
    'market_crash_2008': {
        'description': '2008 Financial Crisis Scenario',
        'market_shock': -0.25,
        'volatility_increase': 3.0,
        'correlation_increase': 0.85
    },
    'covid_crash_2020': {
        'description': 'COVID-19 Market Crash',
        'market_shock': -0.35,
        'volatility_increase': 4.0,
        'correlation_increase': 0.90
    },
    'interest_rate_shock': {
        'description': 'Interest Rate Shock (+300bps)',
        'market_shock': -0.15,
        'volatility_increase': 1.5,
        'correlation_increase': 0.70
    },
    'inflation_spike': {
        'description': 'Inflation Spike Scenario',
        'market_shock': -0.20,
        'volatility_increase': 2.0,
        'correlation_increase': 0.75
    },
    'geopolitical_crisis': {
        'description': 'Geopolitical Crisis',
        'market_shock': -0.30,
        'volatility_increase': 2.5,
        'correlation_increase': 0.80
    }
}

def run_stress_test(returns, portfolio_value, scenario):
    shocked_portfolio_value = portfolio_value * (1 + scenario['market_shock'])
    immediate_loss = portfolio_value - shocked_portfolio_value
    current_vol = returns.std() * np.sqrt(252)
    shocked_vol = current_vol * scenario['volatility_increase']
    np.random.seed(42)
    stressed_returns = np.random.normal(returns.mean() * 0.5, shocked_vol / np.sqrt(252), 30)
    stressed_portfolio_values = [shocked_portfolio_value]
    current_value = shocked_portfolio_value
    for daily_return in stressed_returns:
        current_value *= (1 + daily_return)
        stressed_portfolio_values.append(current_value)
    min_portfolio_value = min(stressed_portfolio_values)
    max_loss = portfolio_value - min_portfolio_value
    max_loss_pct = max_loss / portfolio_value
    final_value = stressed_portfolio_values[-1]
    recovery_needed = (portfolio_value - final_value) / final_value if final_value > 0 else np.inf
    return {
        'immediate_loss': immediate_loss,
        'immediate_loss_pct': immediate_loss / portfolio_value,
        'max_loss': max_loss,
        'max_loss_pct': max_loss_pct,
        'min_portfolio_value': min_portfolio_value,
        'final_portfolio_value': final_value,
        'recovery_needed_pct': recovery_needed,
        'stressed_portfolio_values': stressed_portfolio_values,
        'stressed_vol': shocked_vol
    }

print("🔥 Comprehensive Stress Testing Analysis:")
current_portfolio_value = portfolio_history['Portfolio_Value'].iloc[-1]
stress_test_results = {}
print(f"   Current Portfolio Value: ${current_portfolio_value:,.2f}")
print(f"   Running {len(stress_scenarios)} stress scenarios...\n")
for scenario_name, scenario in stress_scenarios.items():
    print(f"📊 {scenario['description']}:")
    stress_result = run_stress_test(strategy_returns, current_portfolio_value, scenario)
    stress_test_results[scenario_name] = stress_result
    print(f"   Immediate Loss: ${stress_result['immediate_loss']:,.2f} ({stress_result['immediate_loss_pct']:.1%})")
    print(f"   Maximum Loss: ${stress_result['max_loss']:,.2f} ({stress_result['max_loss_pct']:.1%})")
    print(f"   Min Portfolio Value: ${stress_result['min_portfolio_value']:,.2f}")
    print(f"   Recovery Needed: {stress_result['recovery_needed_pct']:.1%}")
    print(f"   Stressed Volatility: {stress_result['stressed_vol']:.1%}\n")

stress_summary = []
for scenario_name, result in stress_test_results.items():
    stress_summary.append({
        'Scenario': stress_scenarios[scenario_name]['description'],
        'Immediate_Loss_Pct': result['immediate_loss_pct'],
        'Max_Loss_Pct': result['max_loss_pct'],
        'Min_Portfolio_Value': result['min_portfolio_value'],
        'Recovery_Needed_Pct': result['recovery_needed_pct'],
        'Stressed_Volatility': result['stressed_vol']
    })
stress_summary_df = pd.DataFrame(stress_summary)
print("📋 Stress Test Summary:")
display(stress_summary_df.round(4))

fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Maximum Loss by Scenario', 'Portfolio Value Under Stress', 'Recovery Requirements', 'Stressed Volatility'),
    vertical_spacing=0.1)
fig.add_trace(go.Bar(x=stress_summary_df['Scenario'], y=stress_summary_df['Max_Loss_Pct'] * 100, name='Max Loss %', marker_color='red'), row=1, col=1)
worst_scenario = stress_summary_df.loc[stress_summary_df['Max_Loss_Pct'].idxmax(), 'Scenario']
worst_scenario_key = [k for k, v in stress_scenarios.items() if v['description'] == worst_scenario][0]
worst_values = stress_test_results[worst_scenario_key]['stressed_portfolio_values']
fig.add_trace(go.Scatter(x=list(range(len(worst_values))), y=worst_values, name=f'Portfolio Value ({worst_scenario})', line=dict(color='red')), row=1, col=2)
fig.add_trace(go.Bar(x=stress_summary_df['Scenario'], y=stress_summary_df['Recovery_Needed_Pct'] * 100, name='Recovery Needed %', marker_color='orange'), row=2, col=1)
fig.add_trace(go.Bar(x=stress_summary_df['Scenario'], y=stress_summary_df['Stressed_Volatility'] * 100, name='Stressed Vol %', marker_color='purple'), row=2, col=2)
fig.update_layout(title='Comprehensive Stress Testing Results', height=600, showlegend=False)
fig.show()

max_acceptable_loss = 0.15
scenarios_exceeding_risk = stress_summary_df[stress_summary_df['Max_Loss_Pct'] > max_acceptable_loss]
print(f"\n⚠️ Risk Appetite Assessment:")
print(f"   Maximum acceptable loss: {max_acceptable_loss:.1%}")
print(f"   Scenarios exceeding risk appetite: {len(scenarios_exceeding_risk)}/{len(stress_summary_df)}")
if len(scenarios_exceeding_risk) > 0:
    print("   Problematic scenarios:")
    for _, scenario in scenarios_exceeding_risk.iterrows():
        print(f"     - {scenario['Scenario']}: {scenario['Max_Loss_Pct']:.1%} loss")
else:
    print("   ✅ All scenarios within risk appetite")


🔥 Comprehensive Stress Testing Analysis:


NameError: name 'portfolio_history' is not defined

## 4. Advanced Factor Risk Models


In [ ]:
print("🔬 Advanced Factor Risk Models Analysis:")
try:
    import yfinance as yf
    factor_data = {
        'Market': yf.download('^GSPC', start='2020-01-01', end='2024-01-01')['Close'].pct_change(),
        'Small_Cap': yf.download('IWM', start='2020-01-01', end='2024-01-01')['Close'].pct_change(),
        'Value': yf.download('IWD', start='2020-01-01', end='2024-01-01')['Close'].pct_change(),
        'Growth': yf.download('IWF', start='2020-01-01', end='2024-01-01')['Close'].pct_change(),
        'Momentum': yf.download('MTUM', start='2020-01-01', end='2024-01-01')['Close'].pct_change(),
        'Quality': yf.download('QUAL', start='2020-01-01', end='2024-01-01')['Close'].pct_change(),
        'Low_Vol': yf.download('USMV', start='2020-01-01', end='2024-01-01')['Close'].pct_change()
    }
    factor_returns = pd.DataFrame(factor_data).dropna()
    aligned_data = pd.concat([strategy_returns.rename('Strategy'), factor_returns], axis=1).dropna()
    if len(aligned_data) > 100:
        from sklearn.linear_model import LinearRegression
        from sklearn.metrics import r2_score
        X = aligned_data[factor_returns.columns]
        y = aligned_data['Strategy']
        factor_model = LinearRegression()
        factor_model.fit(X, y)
        factor_loadings = pd.Series(factor_model.coef_, index=factor_returns.columns)
        alpha = factor_model.intercept_
        r_squared = r2_score(y, factor_model.predict(X))
        print("\n📊 Multi-Factor Model Results:")
        print(f"   Alpha (daily): {alpha:.6f} ({alpha*252:.4f} annualized)")
        print(f"   R-squared: {r_squared:.4f}")
        factor_cov = factor_returns.cov() * 252
        factor_risk_contrib = {}
        total_factor_risk = 0
        for i, f1 in enumerate(factor_returns.columns):
            for j, f2 in enumerate(factor_returns.columns):
                contrib = factor_loadings[f1] * factor_loadings[f2] * factor_cov.loc[f1, f2]
                if i == j:
                    factor_risk_contrib[f1] = contrib
                    total_factor_risk += contrib
                elif i < j:
                    factor_risk_contrib[f"{f1} x {f2}"] = 2 * contrib
                    total_factor_risk += 2 * contrib
        total_strategy_risk = aligned_data['Strategy'].var() * 252
        idiosyncratic_risk = total_strategy_risk - total_factor_risk
        print("\n📊 Risk Decomposition:")
        print(f"   Total Strategy Risk: {total_strategy_risk:.6f} ({np.sqrt(total_strategy_risk):.2%} vol)")
        print(f"   Factor Risk: {total_factor_risk:.6f} ({total_factor_risk/total_strategy_risk:.1%})")
        print(f"   Idiosyncratic Risk: {idiosyncratic_risk:.6f} ({idiosyncratic_risk/total_strategy_risk:.1%})")
        factor_contrib_df = pd.DataFrame({
            'Factor': list(factor_risk_contrib.keys()),
            'Risk_Contribution': list(factor_risk_contrib.values())
        })
        factor_contrib_df['Risk_Contribution_Pct'] = factor_contrib_df['Risk_Contribution'] / total_strategy_risk * 100
        factor_contrib_df = factor_contrib_df.sort_values('Risk_Contribution', key=abs, ascending=False)
        print("\n📋 Factor Risk Contributions:")
        display(factor_contrib_df.head(10).round(4))
        fig = make_subplots(rows=1, cols=2, subplot_titles=('Factor Loadings (Beta)', 'Factor Risk Contributions'), horizontal_spacing=0.1)
        colors = ['red' if x < 0 else 'blue' for x in factor_loadings.values]
        fig.add_trace(go.Bar(x=factor_loadings.index, y=factor_loadings.values, name='Factor Loadings', marker_color=colors), row=1, col=1)
        top_factors = factor_contrib_df[factor_contrib_df['Factor'].isin(factor_returns.columns)].head(7)
        fig.add_trace(go.Bar(x=top_factors['Factor'], y=top_factors['Risk_Contribution_Pct'], name='Risk Contribution %', marker_color='green'), row=1, col=2)
        fig.update_layout(title='Factor Risk Model Analysis', height=400, showlegend=False)
        fig.show()
    else:
        print("   ⚠️ Insufficient aligned data for factor analysis")
except Exception as e:
    print(f"   ❌ Error in factor analysis: {str(e)}")


## 5. GARCH Volatility Modeling and Risk Regimes


In [ ]:
print("📈 GARCH Volatility Modeling and Forecasting:")
garch_results = risk_models.garch_volatility_forecast(strategy_returns, horizon=5)
if 'forecast_volatility' in garch_results and len(garch_results['forecast_volatility']) > 0:
    print("   ✅ GARCH model fitted successfully")
    print(f"   Current volatility: {garch_results['conditional_volatility'].iloc[-1]:.4f}")
    print(f"   5-day forecast volatility: {garch_results['forecast_volatility'][0]:.4f}")
    if 'parameters' in garch_results:
        params = garch_results['parameters']
        print("\n📊 GARCH(1,1) Parameters:")
        for param, value in params.items():
            print(f"   {param}: {value:.6f}")
        if 'alpha[1]' in params and 'beta[1]' in params:
            persistence = params['alpha[1]'] + params['beta[1]']
            print(f"   Persistence (α+β): {persistence:.4f}")
            if persistence < 1:
                half_life = np.log(0.5) / np.log(persistence)
                print(f"   Volatility half-life: {half_life:.1f} days")
    fig = make_subplots(rows=3, cols=1, subplot_titles=('Strategy Returns', 'GARCH Conditional Volatility', 'Volatility Forecast'), vertical_spacing=0.08, row_heights=[0.4, 0.4, 0.2])
    fig.add_trace(go.Scatter(x=strategy_returns.index, y=strategy_returns, name='Returns', line=dict(color='blue', width=1)), row=1, col=1)
    cond_vol = garch_results['conditional_volatility']
    fig.add_trace(go.Scatter(x=cond_vol.index, y=cond_vol, name='Conditional Volatility', line=dict(color='red', width=2)), row=2, col=1)
    fig.add_trace(go.Scatter(x=cond_vol.index, y=cond_vol * 1.5, name='1.5x Vol', line=dict(color='orange', dash='dash'), showlegend=False), row=2, col=1)
    forecast_dates = pd.date_range(start=strategy_returns.index[-1] + pd.Timedelta(days=1), periods=5, freq='D')
    fig.add_trace(go.Scatter(x=forecast_dates, y=garch_results['forecast_volatility'], name='Volatility Forecast', line=dict(color='green', width=3), mode='lines+markers'), row=3, col=1)
    fig.update_layout(title='GARCH Volatility Analysis', height=700, showlegend=True)
    fig.show()
else:
    print("   ⚠️ GARCH model fitting failed, using simple volatility measures")
    rolling_vol_30 = strategy_returns.rolling(30).std() * np.sqrt(252)
    rolling_vol_60 = strategy_returns.rolling(60).std() * np.sqrt(252)
    print(f"   30-day rolling volatility: {rolling_vol_30.iloc[-1]:.2%}")
    print(f"   60-day rolling volatility: {rolling_vol_60.iloc[-1]:.2%}")
    abs_returns = strategy_returns.abs()
    vol_clustering = abs_returns.rolling(10).corr(abs_returns.shift(1)).mean()
    print(f"   Volatility clustering coefficient: {vol_clustering:.4f}")

print("\n🔄 Risk Regime Analysis:")
vol_20day = strategy_returns.rolling(20).std() * np.sqrt(252)
vol_percentiles = vol_20day.quantile([0.33, 0.67])

def classify_regime(vol):
    if pd.isna(vol):
        return 'Unknown'
    elif vol < vol_percentiles.iloc[0]:
        return 'Low Vol'
    elif vol < vol_percentiles.iloc[1]:
        return 'Medium Vol'
    else:
        return 'High Vol'

regimes = vol_20day.apply(classify_regime)
regime_performance = strategy_returns.groupby(regimes).agg({
    'mean': lambda x: x.mean() * 252,
    'std': lambda x: x.std() * np.sqrt(252),
    'count': 'count'
}).round(4)
regime_performance['sharpe'] = regime_performance['mean'] / regime_performance['std']
regime_performance.columns = ['Annual_Return', 'Annual_Volatility', 'Days', 'Sharpe_Ratio']
print("   Volatility regime thresholds:")
print(f"     Low Vol: < {vol_percentiles.iloc[0]:.1%}")
print(f"     Medium Vol: {vol_percentiles.iloc[0]:.1%} - {vol_percentiles.iloc[1]:.1%}")
print(f"     High Vol: > {vol_percentiles.iloc[1]:.1%}")
print("\n📊 Performance by Regime:")
display(regime_performance)
current_regime = regimes.iloc[-1]
current_vol = vol_20day.iloc[-1]
print(f"\n📍 Current Risk Regime: {current_regime} ({current_vol:.1%} volatility)")


## 6. Portfolio Risk Management and Optimization


In [ ]:
print("💼 Portfolio Risk Management and Optimization:")
try:
    portfolio_assets = ['AAPL', 'GOOGL', 'MSFT', 'SPY', 'TLT']
    portfolio_data = {}
    import yfinance as yf
    for asset in portfolio_assets:
        try:
            data = yf.download(asset, start='2020-01-01', end='2024-01-01', progress=False)
            portfolio_data[asset] = data['Close'].pct_change().dropna()
        except:
            continue
    if len(portfolio_data) >= 3:
        portfolio_returns = pd.DataFrame(portfolio_data).dropna()
        print(f"   Portfolio assets: {list(portfolio_returns.columns)}")
        print(f"   Data period: {portfolio_returns.index[0].date()} to {portfolio_returns.index[-1].date()}")
        expected_returns = portfolio_returns.mean() * 252
        covariance_matrix = portfolio_returns.cov() * 252
        print("\n📊 Portfolio Statistics:")
        portfolio_stats = pd.DataFrame({
            'Expected_Return': expected_returns,
            'Volatility': np.sqrt(np.diag(covariance_matrix)),
            'Sharpe_Ratio': expected_returns / np.sqrt(np.diag(covariance_matrix))
        })
        display(portfolio_stats.round(4))
        print("\n🎯 Risk Budgeting Optimization:")
        equal_risk_budgets = pd.Series([1/len(portfolio_returns.columns)] * len(portfolio_returns.columns), index=portfolio_returns.columns)
        rb = risk_models.risk_budgeting_portfolio(expected_returns, covariance_matrix, equal_risk_budgets)
        if rb['optimization_success']:
            optimal_weights = rb['optimal_weights']
            risk_contributions = rb['risk_contributions']
            print(f"   ✅ Risk budgeting optimization successful")
            print(f"   Portfolio return: {rb['portfolio_return']:.2%}")
            print(f"   Portfolio volatility: {rb['portfolio_volatility']:.2%}")
            print("\n   Optimal weights:")
            for asset, weight in optimal_weights.items():
                print(f"     {asset}: {weight:.1%}")
            print("\n   Risk contributions:")
            for asset, contrib in risk_contributions.items():
                print(f"     {asset}: {contrib:.1%}")
        else:
            print("   ❌ Risk budgeting optimization failed")
            optimal_weights = pd.Series([1/len(portfolio_returns.columns)] * len(portfolio_returns.columns), index=portfolio_returns.columns)
        print("\n🔮 Black-Litterman Optimization:")
        market_caps = pd.Series({'AAPL': 3000, 'GOOGL': 2000, 'MSFT': 2800, 'SPY': 1000, 'TLT': 800}).reindex(portfolio_returns.columns).fillna(1000)
        views = {asset: (0.12 if asset in ['AAPL','GOOGL','MSFT'] else 0.08) for asset in portfolio_returns.columns}
        views['TLT'] = 0.03
        bl_result = risk_models.black_litterman_model(market_caps, covariance_matrix, views=views)
        if 'bl_returns' in bl_result:
            print("   ✅ Black-Litterman optimization successful")
            bl_weights = bl_result['optimal_weights']
            bl_returns = bl_result['bl_returns']
            print("\n   Black-Litterman expected returns:")
            for asset, ret in bl_returns.items():
                print(f"     {asset}: {ret:.1%}")
            print("\n   Black-Litterman optimal weights:")
            for asset, weight in bl_weights.items():
                print(f"     {asset}: {weight:.1%}")
        else:
            print("   ❌ Black-Litterman optimization failed")
            bl_weights = optimal_weights
        portfolios = {
            'Equal Weight': pd.Series([1/len(portfolio_returns.columns)] * len(portfolio_returns.columns), index=portfolio_returns.columns),
            'Risk Budgeting': optimal_weights,
            'Black-Litterman': bl_weights
        }
        portfolio_comparison = []
        for name, weights in portfolios.items():
            portfolio_return = weights.T @ expected_returns
            portfolio_var = weights.T @ covariance_matrix @ weights
            portfolio_vol = np.sqrt(portfolio_var)
            portfolio_sharpe = portfolio_return / portfolio_vol
            portfolio_comparison.append({
                'Portfolio': name,
                'Expected_Return': portfolio_return,
                'Volatility': portfolio_vol,
                'Sharpe_Ratio': portfolio_sharpe,
                'Max_Weight': weights.max(),
                'Min_Weight': weights.min()
            })
        portfolio_comparison_df = pd.DataFrame(portfolio_comparison)
        print("\n📊 Portfolio Optimization Comparison:")
        display(portfolio_comparison_df.round(4))
        fig = make_subplots(rows=2, cols=2, subplot_titles=('Portfolio Allocations', 'Risk-Return Profile', 'Portfolio Weights Heatmap', 'Risk Contributions'), specs=[[{"type": "bar"}, {"type": "scatter"}], [{"type": "heatmap"}, {"type": "bar"}]])
        allocation_df = pd.DataFrame(portfolios).T
        for i, portfolio_name in enumerate(allocation_df.index):
            fig.add_trace(go.Bar(x=allocation_df.columns, y=allocation_df.loc[portfolio_name], name=portfolio_name, offsetgroup=i), row=1, col=1)
        fig.add_trace(go.Scatter(x=portfolio_comparison_df['Volatility'], y=portfolio_comparison_df['Expected_Return'], mode='markers+text', text=portfolio_comparison_df['Portfolio'], textposition='top center', marker=dict(size=portfolio_comparison_df['Sharpe_Ratio']*20, color=portfolio_comparison_df['Sharpe_Ratio'], colorscale='Viridis'), name='Portfolios'), row=1, col=2)
        fig.add_trace(go.Heatmap(z=allocation_df.values, x=allocation_df.columns, y=allocation_df.index, colorscale='Blues', name='Weights'), row=2, col=1)
        if rb.get('risk_contributions') is not None:
            fig.add_trace(go.Bar(x=risk_contributions.index, y=risk_contributions.values * 100, name='Risk Contribution %', marker_color='red'), row=2, col=2)
        fig.update_layout(title='Portfolio Risk Management and Optimization', height=800, showlegend=True)
        fig.show()
    else:
        print("   ⚠️ Insufficient assets for portfolio optimization")
except Exception as e:
    print(f"   ❌ Error in portfolio optimization: {str(e)}")


## 7. Risk Monitoring Dashboard and Alerts


In [ ]:
print("🚨 Risk Monitoring Dashboard and Alert System:")
risk_limits = {
    'max_daily_var_95': 0.025,
    'max_portfolio_volatility': 0.20,
    'max_drawdown_limit': 0.15,
    'min_sharpe_ratio': 1.0,
    'max_concentration': 0.40,
    'max_leverage': 1.0,
    'liquidity_threshold': 0.10
}
current_metrics = {
    'current_var_95': abs(var_results[0.95]['historical_var']),
    'current_volatility': strategy_returns.std() * np.sqrt(252),
    'current_drawdown': ((portfolio_history['Portfolio_Value'] / portfolio_history['Portfolio_Value'].expanding().max()) - 1).min(),
    'current_sharpe': (strategy_returns.mean() / strategy_returns.std()) * np.sqrt(252),
    'current_concentration': 1.0,
    'current_leverage': 1.0,
    'current_liquidity': 0.05
}

def generate_risk_alerts(current_metrics, risk_limits):
    alerts = []
    if current_metrics['current_var_95'] > risk_limits['max_daily_var_95']:
        alerts.append({'severity': 'HIGH','metric': 'Value at Risk','current': f"{current_metrics['current_var_95']:.2%}", 'limit': f"{risk_limits['max_daily_var_95']:.2%}", 'message': 'Daily VaR exceeds risk limit'})
    if current_metrics['current_volatility'] > risk_limits['max_portfolio_volatility']:
        alerts.append({'severity': 'MEDIUM','metric': 'Portfolio Volatility','current': f"{current_metrics['current_volatility']:.2%}", 'limit': f"{risk_limits['max_portfolio_volatility']:.2%}", 'message': 'Portfolio volatility above acceptable level'})
    if abs(current_metrics['current_drawdown']) > risk_limits['max_drawdown_limit']:
        alerts.append({'severity': 'HIGH','metric': 'Maximum Drawdown','current': f"{current_metrics['current_drawdown']:.2%}", 'limit': f"{risk_limits['max_drawdown_limit']:.2%}", 'message': 'Portfolio drawdown exceeds risk tolerance'})
    if current_metrics['current_sharpe'] < risk_limits['min_sharpe_ratio']:
        alerts.append({'severity': 'MEDIUM','metric': 'Sharpe Ratio','current': f"{current_metrics['current_sharpe']:.2f}", 'limit': f"{risk_limits['min_sharpe_ratio']:.2f}", 'message': 'Risk-adjusted returns below minimum threshold'})
    if current_metrics['current_concentration'] > risk_limits['max_concentration']:
        alerts.append({'severity': 'MEDIUM','metric': 'Position Concentration','current': f"{current_metrics['current_concentration']:.1%}", 'limit': f"{risk_limits['max_concentration']:.1%}", 'message': 'Portfolio concentration risk detected'})
    if current_metrics['current_liquidity'] < risk_limits['liquidity_threshold']:
        alerts.append({'severity': 'LOW','metric': 'Liquidity','current': f"{current_metrics['current_liquidity']:.1%}", 'limit': f"{risk_limits['liquidity_threshold']:.1%}", 'message': 'Low liquidity buffer detected'})
    return alerts

risk_alerts = generate_risk_alerts(current_metrics, risk_limits)
print(f"\n🚨 Risk Alert Summary ({len(risk_alerts)} alerts):")
if risk_alerts:
    alert_df = pd.DataFrame(risk_alerts)
    severity_colors = {'HIGH': '🔴', 'MEDIUM': '🟡', 'LOW': '🟢'}
    for _, alert in alert_df.iterrows():
        color = severity_colors.get(alert['severity'], '⚪')
        print(f"   {color} {alert['severity']}: {alert['metric']}")
        print(f"      Current: {alert['current']} | Limit: {alert['limit']}")
        print(f"      Message: {alert['message']}")
        print()
else:
    print("   ✅ All risk metrics within acceptable limits")

risk_dashboard = pd.DataFrame({
    'Risk Metric': ['Daily VaR (95%)','Annual Volatility','Maximum Drawdown','Sharpe Ratio','Position Concentration','Liquidity Buffer'],
    'Current Value': [
        f"{current_metrics['current_var_95']:.2%}",
        f"{current_metrics['current_volatility']:.2%}",
        f"{current_metrics['current_drawdown']:.2%}",
        f"{current_metrics['current_sharpe']:.2f}",
        f"{current_metrics['current_concentration']:.1%}",
        f"{current_metrics['current_liquidity']:.1%}"
    ],
    'Risk Limit': [
        f"{risk_limits['max_daily_var_95']:.2%}",
        f"{risk_limits['max_portfolio_volatility']:.2%}",
        f"{risk_limits['max_drawdown_limit']:.2%}",
        f"{risk_limits['min_sharpe_ratio']:.2f}",
        f"{risk_limits['max_concentration']:.1%}",
        f"{risk_limits['liquidity_threshold']:.1%}"
    ],
    'Status': [
        '🔴 BREACH' if current_metrics['current_var_95'] > risk_limits['max_daily_var_95'] else '✅ OK',
        '🔴 BREACH' if current_metrics['current_volatility'] > risk_limits['max_portfolio_volatility'] else '✅ OK',
        '🔴 BREACH' if abs(current_metrics['current_drawdown']) > risk_limits['max_drawdown_limit'] else '✅ OK',
        '🔴 BREACH' if current_metrics['current_sharpe'] < risk_limits['min_sharpe_ratio'] else '✅ OK',
        '🔴 BREACH' if current_metrics['current_concentration'] > risk_limits['max_concentration'] else '✅ OK',
        '🔴 BREACH' if current_metrics['current_liquidity'] < risk_limits['liquidity_threshold'] else '✅ OK'
    ]
})
print("\n📊 Risk Dashboard:")
display(risk_dashboard)

score_breakdown = pd.DataFrame({
    'Component': ['VaR', 'Volatility', 'Drawdown', 'Sharpe', 'Concentration', 'Liquidity'],
    'Score': [
        min(25, (current_metrics['current_var_95'] / risk_limits['max_daily_var_95']) * 25),
        min(20, (current_metrics['current_volatility'] / risk_limits['max_portfolio_volatility']) * 20),
        min(25, (abs(current_metrics['current_drawdown']) / risk_limits['max_drawdown_limit']) * 25),
        max(0, 15 - (current_metrics['current_sharpe'] / risk_limits['min_sharpe_ratio']) * 15),
        min(10, (current_metrics['current_concentration'] / risk_limits['max_concentration']) * 10),
        max(0, 5 - (current_metrics['current_liquidity'] / risk_limits['liquidity_threshold']) * 5)
    ],
    'Max_Score': [25, 20, 25, 15, 10, 5]
})
score_breakdown['Percentage'] = score_breakdown['Score'] / score_breakdown['Max_Score'] * 100

fig = make_subplots(rows=2, cols=2, subplot_titles=('Risk Score Breakdown', 'Risk Limits vs Current', 'Portfolio Risk Over Time', 'Alert Status'), specs=[[{"type": "bar"}, {"type": "bar"}], [{"type": "scatter"}, {"type": "pie"}]])
fig.add_trace(go.Bar(x=score_breakdown['Component'], y=score_breakdown['Percentage'], name='Risk Score %', marker_color=['red' if s > 80 else 'orange' if s > 50 else 'green' for s in score_breakdown['Percentage']]), row=1, col=1)
metrics_values = [current_metrics['current_var_95']*100, current_metrics['current_volatility']*100, abs(current_metrics['current_drawdown'])*100]
limits_values = [risk_limits['max_daily_var_95']*100, risk_limits['max_portfolio_volatility']*100, risk_limits['max_drawdown_limit']*100]
metric_names = ['VaR %', 'Volatility %', 'Drawdown %']
fig.add_trace(go.Bar(x=metric_names, y=metrics_values, name='Current', marker_color='blue', opacity=0.7), row=1, col=2)
fig.add_trace(go.Bar(x=metric_names, y=limits_values, name='Limit', marker_color='red', opacity=0.5), row=1, col=2)
window_size = 60
rolling_var = []
for i in range(window_size, len(strategy_returns)):
    window_returns = strategy_returns.iloc[i-window_size:i]
    var_60d = np.percentile(window_returns, 5)
    rolling_var.append(-var_60d)
rolling_var_dates = strategy_returns.index[window_size:]
if len(rolling_var) > 0:
    fig.add_trace(go.Scatter(x=rolling_var_dates, y=rolling_var, name='Rolling VaR', line=dict(color='purple')), row=2, col=1)
    fig.add_hline(y=risk_limits['max_daily_var_95'], line_dash='dash', line_color='red', row=2, col=1)
alert_counts = {'No Alerts': 0, 'Low': 0, 'Medium': 0, 'High': 0}
if risk_alerts:
    for alert in risk_alerts:
        alert_counts[alert['severity'].title()] += 1
else:
    alert_counts['No Alerts'] = 1
alert_counts = {k: v for k, v in alert_counts.items() if v > 0}
fig.add_trace(go.Pie(labels=list(alert_counts.keys()), values=list(alert_counts.values()), name='Alert Status'), row=2, col=2)
fig.update_layout(title='Risk Monitoring Dashboard', height=800, showlegend=True)
fig.show()
print("\n📋 Risk Score Component Breakdown:")
display(score_breakdown.round(2))


## 8. Regulatory Reporting and Compliance


In [ ]:
print("📋 Regulatory Reporting and Compliance:")

def calculate_basel_iii_metrics(returns, portfolio_value):
    es_975 = portfolio_risk_manager.conditional_var(returns, 0.975)
    stressed_returns = returns * 1.5
    stressed_var = portfolio_risk_manager.historical_var(stressed_returns, 0.99)
    annual_var = np.sqrt(252) * portfolio_risk_manager.historical_var(returns, 0.99)
    var_99 = portfolio_risk_manager.historical_var(returns, 0.99)
    market_risk_capital = max(var_99 * portfolio_value * 3, es_975 * portfolio_value * 2.5)
    return {
        'expected_shortfall_975': es_975,
        'stressed_var_99': stressed_var,
        'annual_var_99': annual_var,
        'market_risk_capital': market_risk_capital,
        'capital_ratio': market_risk_capital / portfolio_value
    }

basel_metrics = calculate_basel_iii_metrics(strategy_returns, portfolio_history['Portfolio_Value'].iloc[-1])
print("\n🏛️ Basel III Risk Measures:")
print(f"   Expected Shortfall (97.5%): {basel_metrics['expected_shortfall_975']:.4f}")
print(f"   Stressed VaR (99%): {basel_metrics['stressed_var_99']:.4f}")
print(f"   Annual VaR (99%): {basel_metrics['annual_var_99']:.4f}")
print(f"   Market Risk Capital: ${basel_metrics['market_risk_capital']:,.2f}")
print(f"   Capital Ratio: {basel_metrics['capital_ratio']:.2%}")

regulatory_report = {
    'report_date': datetime.now().strftime('%Y-%m-%d'),
    'reporting_entity': 'Quantitative Trading Strategy',
    'portfolio_value': portfolio_history['Portfolio_Value'].iloc[-1],
    'base_currency': 'USD',
    'risk_measures': {
        'var_95_1d': var_results[0.95]['historical_var'],
        'var_99_1d': var_results[0.99]['historical_var'],
        'es_97.5_1d': basel_metrics['expected_shortfall_975'],
        'stressed_var_99_1d': basel_metrics['stressed_var_99']
    },
    'portfolio_metrics': {
        'volatility_annual': current_metrics['current_volatility'],
        'sharpe_ratio': current_metrics['current_sharpe'],
        'max_drawdown': current_metrics['current_drawdown'],
        'beta_to_market': 1.0
    },
    'limit_monitoring': {
        'var_limit_utilization': current_metrics['current_var_95'] / risk_limits['max_daily_var_95'],
        'volatility_limit_utilization': current_metrics['current_volatility'] / risk_limits['max_portfolio_volatility'],
        'drawdown_limit_utilization': abs(current_metrics['current_drawdown']) / risk_limits['max_drawdown_limit']
    },
    'stress_test_results': {
        'worst_case_loss': max([result['max_loss_pct'] for result in stress_test_results.values()]),
        'worst_scenario': max(stress_test_results.keys(), key=lambda k: stress_test_results[k]['max_loss_pct'])
    },
    'compliance_status': {
        'var_compliant': current_metrics['current_var_95'] <= risk_limits['max_daily_var_95'],
        'volatility_compliant': current_metrics['current_volatility'] <= risk_limits['max_portfolio_volatility'],
        'drawdown_compliant': abs(current_metrics['current_drawdown']) <= risk_limits['max_drawdown_limit'],
        'overall_compliant': len(risk_alerts) == 0
    }
}

regulatory_summary = pd.DataFrame({
    'Risk Measure': ['VaR (95%, 1-day)','VaR (99%, 1-day)','Expected Shortfall (97.5%, 1-day)','Stressed VaR (99%, 1-day)','Annual Volatility','Maximum Drawdown','Market Risk Capital'],
    'Value': [
        f"{regulatory_report['risk_measures']['var_95_1d']:.4f}",
        f"{regulatory_report['risk_measures']['var_99_1d']:.4f}",
        f"{regulatory_report['risk_measures']['es_97.5_1d']:.4f}",
        f"{regulatory_report['risk_measures']['stressed_var_99_1d']:.4f}",
        f"{regulatory_report['portfolio_metrics']['volatility_annual']:.2%}",
        f"{regulatory_report['portfolio_metrics']['max_drawdown']:.2%}",
        f"${basel_metrics['market_risk_capital']:,.0f}"
    ],
    'Basel III Compliant': ['✅', '✅', '✅', '✅', 'N/A', 'N/A', '✅']
})
print("\n📊 Regulatory Risk Summary:")
display(regulatory_summary)
compliance_status = regulatory_report['compliance_status']
print("\n✅ Compliance Status:")
print(f"   VaR Compliant: {'✅ YES' if compliance_status['var_compliant'] else '❌ NO'}")
print(f"   Volatility Compliant: {'✅ YES' if compliance_status['volatility_compliant'] else '❌ NO'}")
print(f"   Drawdown Compliant: {'✅ YES' if compliance_status['drawdown_compliant'] else '❌ NO'}")
print(f"   Overall Compliant: {'✅ YES' if compliance_status['overall_compliant'] else '❌ NO'}")

import json
with open('../data/regulatory_risk_report.json', 'w') as f:
    def convert_numpy(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return obj
    clean_report = json.loads(json.dumps(regulatory_report, default=convert_numpy))
    json.dump(clean_report, f, indent=2)
print("\n💾 Regulatory report exported to: ../data/regulatory_risk_report.json")


## 9. Final Risk Assessment and Recommendations


In [ ]:
print("📋 Comprehensive Risk Assessment Summary:")
assessment_scores = {
    'market_risk': 'MEDIUM' if current_metrics['current_volatility'] > 0.15 else 'LOW',
    'liquidity_risk': 'HIGH' if current_metrics['current_liquidity'] < 0.05 else 'MEDIUM' if current_metrics['current_liquidity'] < 0.10 else 'LOW',
    'concentration_risk': 'HIGH' if current_metrics['current_concentration'] > 0.5 else 'MEDIUM' if current_metrics['current_concentration'] > 0.3 else 'LOW',
    'operational_risk': 'LOW',
    'model_risk': 'MEDIUM' if len([a for a in risk_alerts if a['severity'] == 'HIGH']) > 0 else 'LOW',
    'regulatory_risk': 'LOW' if regulatory_report['compliance_status']['overall_compliant'] else 'HIGH'
}
print("\n🎯 Risk Category Assessment:")
for risk_type, level in assessment_scores.items():
    emoji = '🔴' if level == 'HIGH' else '🟡' if level == 'MEDIUM' else '🟢'
    print(f"   {emoji} {risk_type.replace('_', ' ').title()}: {level}")

high_risks = sum(1 for level in assessment_scores.values() if level == 'HIGH')
medium_risks = sum(1 for level in assessment_scores.values() if level == 'MEDIUM')
if high_risks >= 2:
    overall_risk = 'HIGH'
elif high_risks == 1 or medium_risks >= 3:
    overall_risk = 'MEDIUM'
else:
    overall_risk = 'LOW'
print(f"\n🎯 Overall Risk Rating: {overall_risk}")
print(f"   Risk Score (composite): {score_breakdown['Percentage'].mean():.1f}/100")
print(f"   High Risk Areas: {high_risks}")
print(f"   Medium Risk Areas: {medium_risks}")

recommendations = []
if current_metrics['current_var_95'] > risk_limits['max_daily_var_95']:
    recommendations.append({'priority': 'HIGH','category': 'Risk Reduction','action': 'Reduce position sizes to bring VaR within limits','timeline': 'Immediate'})
if current_metrics['current_volatility'] > 0.20:
    recommendations.append({'priority': 'MEDIUM','category': 'Volatility Management','action': 'Implement dynamic position sizing based on volatility regime','timeline': '1 week'})
if current_metrics['current_concentration'] > 0.4:
    recommendations.append({'priority': 'MEDIUM','category': 'Diversification','action': 'Increase diversification across assets and strategies','timeline': '2 weeks'})
if current_metrics['current_sharpe'] < 1.0:
    recommendations.append({'priority': 'MEDIUM','category': 'Performance Enhancement','action': 'Review and optimize strategy parameters','timeline': '1 month'})
worst_stress_loss = max([result['max_loss_pct'] for result in stress_test_results.values()])
if worst_stress_loss > 0.25:
    recommendations.append({'priority': 'HIGH','category': 'Stress Testing','action': 'Implement additional hedging for extreme market scenarios','timeline': 'Immediate'})
recommendations.append({'priority': 'LOW','category': 'Risk Monitoring','action': 'Implement real-time risk monitoring and automated alerts','timeline': '1 month'})
recommendations.append({'priority': 'LOW','category': 'Model Validation','action': 'Conduct quarterly model backtesting and validation','timeline': 'Ongoing'})
recommendations_df = pd.DataFrame(recommendations)
print("\n📋 Risk Management Recommendations:")
if not recommendations_df.empty:
    display(recommendations_df)
    high_priority = recommendations_df[recommendations_df['priority'] == 'HIGH']
    medium_priority = recommendations_df[recommendations_df['priority'] == 'MEDIUM']
    print(f"\n⚡ Immediate Actions Required ({len(high_priority)} items):")
    for _, rec in high_priority.iterrows():
        print(f"   🔴 {rec['category']}: {rec['action']}")
    print(f"\n📅 Short-term Actions ({len(medium_priority)} items):")
    for _, rec in medium_priority.iterrows():
        print(f"   🟡 {rec['category']}: {rec['action']}")

final_risk_assessment = {
    'assessment_date': datetime.now().isoformat(),
    'overall_risk_rating': overall_risk,
    'risk_score': score_breakdown['Percentage'].mean(),
    'risk_categories': assessment_scores,
    'key_metrics': current_metrics,
    'risk_limits': risk_limits,
    'alerts': risk_alerts,
    'stress_test_summary': {
        'worst_case_loss': worst_stress_loss,
        'scenarios_tested': len(stress_scenarios)
    },
    'basel_iii_metrics': basel_metrics,
    'recommendations': recommendations,
    'compliance_status': regulatory_report['compliance_status']
}
with open('../data/final_risk_assessment.pkl', 'wb') as f:
    pickle.dump(final_risk_assessment, f)
if not recommendations_df.empty:
    recommendations_df.to_csv('../data/risk_recommendations.csv', index=False)
print("\n💾 Final risk assessment saved to: ../data/final_risk_assessment.pkl")
print("💾 Recommendations saved to: ../data/risk_recommendations.csv")

executive_risk_summary = f"""
📊 EXECUTIVE RISK SUMMARY
Report Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}

🎯 OVERALL ASSESSMENT: {overall_risk} RISK
Risk Score: {score_breakdown['Percentage'].mean():.0f}/100

📈 KEY METRICS:
• Daily VaR (95%): {current_metrics['current_var_95']:.2%}
• Annual Volatility: {current_metrics['current_volatility']:.1%}
• Maximum Drawdown: {current_metrics['current_drawdown']:.1%}
• Sharpe Ratio: {current_metrics['current_sharpe']:.2f}
• Portfolio Value: ${portfolio_history['Portfolio_Value'].iloc[-1]:,.0f}

🚨 ACTIVE ALERTS: {len(risk_alerts)}
• High Priority: {len([a for a in risk_alerts if a['severity'] == 'HIGH'])}
• Medium Priority: {len([a for a in risk_alerts if a['severity'] == 'MEDIUM'])}
• Low Priority: {len([a for a in risk_alerts if a['severity'] == 'LOW'])}

🔥 STRESS TEST RESULTS:
• Worst Case Loss: {worst_stress_loss:.1%}
• Scenarios Tested: {len(stress_scenarios)}
• Market Risk Capital: ${basel_metrics['market_risk_capital']:,.0f}

✅ COMPLIANCE STATUS: {'COMPLIANT' if regulatory_report['compliance_status']['overall_compliant'] else 'NON-COMPLIANT'}

📋 IMMEDIATE ACTIONS: {len(high_priority) if 'high_priority' in locals() else 0}
"""
print(executive_risk_summary)
with open('../data/executive_risk_summary.txt', 'w') as f:
    f.write(executive_risk_summary)
print("\n💾 Executive summary saved to: ../data/executive_risk_summary.txt")
print("\n🎉 Comprehensive risk assessment complete!")
